In [190]:
pip install pydantic

Initializing Incident Schema

In [191]:
from pydantic import BaseModel, Field, field_validator
from typing import  List, Optional
import json

In [192]:
class MicroServiceComponent(BaseModel):
  service_name: str = Field(description="The exact name of the backend component casuing issues")
  environment: str = Field(description="Must be out of these options in lowercase: 'production','staging','development'.")

class SystemIncidentSchema(BaseModel):
  incident_id: str = Field(description="Unique tracking id formatted exactly as INC-YYYY-XXXXX")
  severity_level: str = Field(description="Priority tier. Must be uppercase: 'CRITICAL','MAJOR','MINOR'")
  main_reason: str = Field(description="A brief, but high-level technical summary of why the crash happened")
  impacted_services: List[MicroServiceComponent] = Field(description="List of all the services affected by the crash")
  est_downtime_min: int = Field(description="Integer tracking the duration of service disruption")


  # Adding a validator to check for common LLM formatting misses

  @field_validator('severity_level')
  @classmethod
  def validate_severity(cls,value: str) -> str:
    if value not in ['CRITICAL','MAJOR','MINOR']:
      raise ValueError('Severity must be one of CRITICAL, MAJOR, MINOR')
    return value

In [193]:
# unstructured_logs = [
#     "LOG ENTRY: 14:22 UTC. auth-gateway microservice crashed in production. Database connection pool exhausted due to peak traffic spike. We need an incident log immediately. Let's trace this under INC-2026-99211. Total recovery took about 45 minutes. Mark it down as highly severe.",
#     "ALERT MESSAGE: billing-engine component failed in staging environment. Payment gateway webhooks dropped a series of transactional post-requests between 02:00 and 02:20 AM. Set tracking ID to INC-2026-00412. Downtime duration estimated around 20 minutes. Severity tier is MAJOR.",
#     "CHAT TRANSCRIPT: hey team, checking the logs for the dev environment deployment. the user-profile service kept throwing 500 internal errors during the migration check. it was acting up for about 10 minutes before self-healing. reference ID: INC-2026-77312. Keep it categorized as MINOR impact."
# ]

unstructured_logs = [
    """
    [2026-07-16T14:22:11.892Z] [NODE-04A] [KERNEL-OOM] [SEV: CRITICAL]
    *** INFRASTRUCTURE ALERT DISPATCH ***
    At approximately 14:20 UTC, the core 'auth-gateway' microservice instance running on pod auth-gw-7ff8bc894-x29wl crashed violently in the production cluster (us-east-1).
    Telemetry analysis indicates a severe heap memory leak within the JWT validation routine, which caused JVM memory utilization to instantly spike from a baseline of 42% to 100%, triggering an OS-level Out-Of-Memory (OOM) killer event.

    CRITICAL CASCADE: Because the auth-gateway went dark, the downstream 'user-profile' and 'inventory-service' instances immediately entered a connection-wait loop. This thread pool exhaustion propagated upstream to our cloud load balancers, resulting in widespread 503 Service Unavailable errors for 84,200 active users.

    INCIDENT ROOM SUMMARY:
    - Internal Ticket Assignment: INC-2026-99211
    - Root Cause Category: Memory Leak / Thread Starvation
    - Mitigation Strategy: Site Reliability Engineering (SRE) forced a rolling restart of the auth deployment and temporarily scaled the horizontal pod autoscaler (HPA) minimum to 15 nodes.
    - Timeline Details: System went dark at 14:22 UTC. Full service restoration and traffic stabilization achieved at 15:07 UTC. Total active outage/recovery duration: exactly 45 minutes.
    - Audit Classification: Tier-1 Infrastructure Failure. Status: Resolved.
    """,
    """
    ==> COMPONENT FAULT LOG | ENVIRONMENT: STAGING | REGION: eu-west-2
    TIMESTAMP: Thu Jul 16 02:00:45 UTC 2026
    SOURCE: billing-engine (v4.12.2-rc3)

    [ERROR-CODE: ERR_DB_DEADLOCK_TRANSACTION]
    An automated alert triggered following a series of dropped payment transactional post-requests. The staging PostgreSQL replica (db-billing-stg-01) entered a severe deadlock loop during a concurrent bulk invoice processing job.
    As a direct consequence, the asynchronous event processor was unable to acknowledge incoming payment gateway webhooks from Stripe and Adyen.

    DROPPED MESSAGES AUDIT:
    - Webhook failures began dropping at precisely 02:00 AM UTC.
    - Webhook processing resumed normalcy at 02:20 AM UTC after SRE terminated the blocking transaction pid (PID: 28411).
    - Dropped event count: 4,120 transaction packets requires manual ingestion scripts.

    MANAGEMENT METADATA:
    - Operational Reference Token: INC-2026-00412
    - Operational Impact Severity Assessment: MAJOR (Staging Data Inconsistency)
    - Downtime Window: Calculated to be precisely 20 minutes.
    - Follow-up Action: Database index optimization required on the `invoice_status_ledger` table.
    """,
    """
    Slack Dev-Ops Chat Export -- #war-room-dev-migration (Date: 2026-07-15)
    ----------------------------------------------------------------------
    [11:02:15] @sarah_sre: "Hey team, seeing some weird anomalies during our database migration checks on the dev-cluster-v2."
    [11:03:40] @infra_bot [APP]: "[ALERT] 'user-profile' microservice is experiencing an elevated error rate. Current metrics show 42% of incoming requests are failing with HTTP 500 Internal Server Errors."
    [11:05:12] @dave_backend: "Investigating now. Ah, look at this. The migration script locked the `user_metadata` table briefly while applying the new schema patch. The app code didn't have an intelligent retry policy configured for the db connection."
    [11:11:00] @sarah_sre: "Looks like the script finished running and released the lock. The user-profile service is self-healing now. Error rates dropped back to 0%. It was acting up for roughly 10 minutes total."
    [11:12:30] @dave_backend: "Perfect. I'll log it in Jira just so we track the telemetry behavior. I'll flag it under tracking token INC-2026-77312. Since it's isolated to the non-production development cluster and recovered on its own without data loss, we can comfortably categorize this as a MINOR impact event."
    ----------------------------------------------------------------------
    """,
    """
    [NET-CORE] [2026-07-14 18:40:22 MST] -- PERIMETER ROUTER FAULT DETECTION
    Anomalous BGP path flapping detected at our core Edge Routing Facility (DC-WEST-02). A misconfigured route map update broadcasted by an upstream transit provider caused an internal Border Gateway Protocol (BGP) routing loop.

    NETWORK CONSEQUENCES:
    Packets bouncing between router-core-01a and router-core-02b rapidly exceeded their Time-To-Live (TTL) limits. This created a complete network partition inside our western availability zone, completely isolating our 'media-streaming-pipeline' microservice from the global content delivery network (CDN).

    SRE ACTION REPORT:
    Engineers immediately issued a manual prefix-list override and withdrew our BGP routes from the faulty upstream path, forcing traffic to fall back entirely to our secondary fiber backbone channel.

    METRICS & AUDITING:
    - Incident Record Locator: INC-2026-88104
    - Outage Boundary: Started 18:40 MST, stable mitigation verified at 20:15 MST.
    - Outage Duration: 1 hour and 35 minutes (95 minutes total).
    - Impact Severity Tier: CRITICAL (Complete regional availability failure)
    """,
    """
    {
      "log_source": "api-gateway-v2",
      "environment": "production",
      "payload": {
        "timestamp": "2026-07-13T23:55:00Z",
        "sys_message": "Cascading cache miss anomaly detected. The Redis cluster 'cache-prod-cluster' suffered a localized split-brain event due to a transient network blip. When the cache nodes re-synchronized, the keyspace containing our API rate-limiting rules was completely flushed.",
        "consequences": "This caused a massive 'Cache Stampede' or invalidation storm. Over 150 downstream worker services simultaneously hit our primary relational database (`prod-customer-master`) to fetch authorization credentials, causing DB CPU utilization to hit 100% instantly.",
        "throttling": "To protect the database from crashing, the API Gateway immediately engaged a global emergency throttling mechanism, rejecting 65% of all inbound public API calls with HTTP 429 Too Many Requests.",
        "resolution": "The cache cluster data was manually re-seeded from replica nodes, and database connections cooled down. The outage began at 23:55 UTC and normal operations resumed at 00:30 UTC the next day.",
        "tracking": {
          "ticket_id": "INC-2026-11522",
          "calculated_downtime_minutes": 35,
          "severity_level": "MAJOR"
        }
      }
    }
    """
]

print(f"Loaded {len(unstructured_logs)} unstructured logs for agent validation testing.")

Loaded 5 unstructured logs for agent validation testing.


In [194]:
import random
from datetime import datetime, timedelta

print("--- Issue 5.5: Generating Synthetic Enterprise Data Batch ---")

def generate_chaotic_log():
    services = ['auth-gateway', 'billing-engine', 'user-profile-db', 'search-indexer', 'notification-service', 'inventory-api', 'redis-cache']
    envs = ['production', 'staging', 'development', 'PROD', 'dev', 'stage']

    # Intentionally mixing correct and incorrect severity labels to trigger Pydantic self-correction
    severities = ['CRITICAL', 'MAJOR', 'MINOR', 'highly severe', 'level 1', 'low impact', 'Sev-2', 'catastrophic', 'minor issue']

    causes = [
        'OOM killer terminated the pod',
        'DDoS attack saturated bandwidth',
        'bad config push by DevOps',
        'database connection pool exhaustion',
        'expired SSL certificate',
        'third-party API rate limit exceeded'
    ]

    templates = [
        # Template 1: Messy Slack Message
        "SLACK (DevOps Channel): hey guys, {service} just went down in {env}. looks like {cause}. i'm tracking this as INC-2026-{id}. took {downtime} mins to fix. severity is {severity}.",

        # Template 2: Automated PagerDuty Alert
        "PAGERDUTY ALERT: {env} environment degraded. Service impacted: {service}. Root cause diagnostics: {cause}. Resolution time: {downtime} minutes. Priority: {severity}. Ref: INC-2026-{id}.",

        # Template 3: Raw Syslog
        "SYS_LOG [14:22 UTC] - FATAL ERROR in {service} [{env}]. Trace: {cause}. Downtime tracked at {downtime}m. Ticket INC-2026-{id} opened. Impact level assessed as {severity}."
    ]

    return random.choice(templates).format(
        service=random.choice(services),
        env=random.choice(envs),
        cause=random.choice(causes),
        id=random.randint(10000, 99999),
        downtime=random.randint(5, 120),
        severity=random.choice(severities)
    )

# Generate a massive batch of 30 unique, chaotic logs
enterprise_logs_batch = [generate_chaotic_log() for _ in range(30)]

print(f"✅ Successfully generated {len(enterprise_logs_batch)} chaotic incident logs.")
print("\nSample Log 1:", enterprise_logs_batch[0])
print("Sample Log 2:", enterprise_logs_batch[1])

--- Issue 5.5: Generating Synthetic Enterprise Data Batch ---
✅ Successfully generated 30 chaotic incident logs.

Sample Log 1: SYS_LOG [14:22 UTC] - FATAL ERROR in inventory-api [dev]. Trace: bad config push by DevOps. Downtime tracked at 100m. Ticket INC-2026-45606 opened. Impact level assessed as catastrophic.
Sample Log 2: SYS_LOG [14:22 UTC] - FATAL ERROR in auth-gateway [development]. Trace: bad config push by DevOps. Downtime tracked at 80m. Ticket INC-2026-28001 opened. Impact level assessed as CRITICAL.


In [195]:
!pip install -q datasets

from datasets import load_dataset
import random

print("--- Issue 5.5: Loading REAL Industry IT Incident Reports ---")

# Load a massive open-source dataset of real GitHub issues and bug reports
print("Downloading real-world dataset from Hugging Face...")
dataset = load_dataset("lewtun/github-issues", split="train")

# Filter for logs that have enough text to be chaotic and challenging for the AI
messy_issues = [row['body'] for row in dataset if row['body'] and len(row['body']) > 300]
print(len(messy_issues))
# Sample 50 random real-world incident reports for our batch process
enterprise_logs_batch = random.sample(messy_issues, 30)

print(f"✅ Successfully loaded {len(enterprise_logs_batch)} unstructured IT incident logs.")
print("\n--- Sample Log 1 (Notice how chaotic real data is!) ---")
print(enterprise_logs_batch[0][:800] + "...\n")

--- Issue 5.5: Loading REAL Industry IT Incident Reports ---


Repo card metadata block was not found. Setting CardData to empty.


1413
✅ Successfully loaded 30 unstructured IT incident logs.

--- Sample Log 1 (Notice how chaotic real data is!) ---
Hello :),

I have been trying to load data using a JSON file. Based on the [docs](https://huggingface.co/docs/datasets/loading_datasets.html#json-files), the following format is supported:

```json
{"key1":11, "key2":12, "key3":13}
{"key1":21, "key2":22, "key3":23}
```
 But, when I try loading a dataset with the same format, I get a JSONDecodeError : `JSONDecodeError: Extra data: line 2 column 1 (char 7142)`. Now, this is expected when using `json` to load a JSON file. But I was wondering if there are any special arguments to pass when using `load_dataset` as the docs suggest that this format is supported.

When I convert the JSON file to a list of dictionaries format, I get AttributeError: `AttributeError: 'list' object has no attribute 'keys'`. So, I can't convert them to list ...



Setting Up Gemini Generative AI Framework

In [196]:
# !pip install -q -U google-genai
!pip install -q -U groq

In [197]:
# from google import genai
# from google.colab import userdata
# import os

from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client =Groq(api_key=GROQ_API_KEY)

In [198]:
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# client = genai.Client(api_key=GEMINI_API_KEY)


# for model in client.models.list():
#   # print(model)
#   print(f"Model ID: {model.name} -> {model.display_name}")
# # model = genai.GenerativeModel('gemini-3.5-flash')

In [199]:
def generate_extraction_prompt(targetlog_text: str)->str:
  schema = SystemIncidentSchema.model_json_schema()

  prompt = f"""
  You are am automated backend insfrastructure logs processing model.
  Your task is to extract structural telemetry data from raw unstructured input logs.

  OUTPUT: Your output must be a raw, valid JSON string that maps  precisely to this JSON schema:
  {json.dumps(schema,indent=2)}

  Return only valid, parseable JSON string. Do not wrap it in markdown code blocks like``` json ...```
  Do not add extra useless fluff.

  Raw Log to Process:
  {targetlog_text}
  """
  return prompt

# sample = generate_extraction_prompt(unstructured_logs[0])
sample = generate_extraction_prompt(enterprise_logs_batch[0])
print(f"Sample Prompt: {sample}")

Sample Prompt: 
  You are am automated backend insfrastructure logs processing model.
  Your task is to extract structural telemetry data from raw unstructured input logs.

  OUTPUT: Your output must be a raw, valid JSON string that maps  precisely to this JSON schema:
  {
  "$defs": {
    "MicroServiceComponent": {
      "properties": {
        "service_name": {
          "description": "The exact name of the backend component casuing issues",
          "title": "Service Name",
          "type": "string"
        },
        "environment": {
          "description": "Must be out of these options in lowercase: 'production','staging','development'.",
          "title": "Environment",
          "type": "string"
        }
      },
      "required": [
        "service_name",
        "environment"
      ],
      "title": "MicroServiceComponent",
      "type": "object"
    }
  },
  "properties": {
    "incident_id": {
      "description": "Unique tracking id formatted exactly as INC-YYYY-XXXXX

In [200]:
print("Validation Check")
# response = client.models.generate_content(model='gemini-1.5-flash',contents=sample)

response = client.chat.completions.create(
  # model="llama3-8b-8192", decommissioned
  model = "openai/gpt-oss-20b",
  messages=[
      {"role":"system","content": "You are an automated backend infrastructure logs processing model."},
      {"role":"user","content": sample}
  ]
)
print(f"Response: {response}")
print(f"{clean_json_string(response.choices[0].message.content)}")

Validation Check
Response: ChatCompletion(id='chatcmpl-d7974dd7-1017-4acc-bfe6-2483fe82c9af', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"incident_id":"INC-0000-00000","severity_level":"MINOR","main_reason":"User encountered JSONDecodeError when loading dataset, likely due to multi-line JSON format.","impacted_services":[],"est_downtime_min":0}', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to parse raw log: It\'s a question from user about loading JSON file in huggingface datasets, encountering JSONDecodeError, expecting that the format is supported. It\'s not an incident. There\'s no crash, no incident. The schema requires certain fields. But there\'s no incident. So likely we cannot produce a valid JSON for schema. But maybe we should output an empty or minimal? The instructions say to output JSON mapping to the schema. But we can\'t produce data that doesn\'t exist. Perhaps

In [201]:
print(f"Response: {response.choices[0].message.content}")

Response: {"incident_id":"INC-0000-00000","severity_level":"MINOR","main_reason":"User encountered JSONDecodeError when loading dataset, likely due to multi-line JSON format.","impacted_services":[],"est_downtime_min":0}


Self Correction

In [202]:
from pydantic import ValidationError

In [203]:
def clean_json_string(raw:str)->str:
  cleaned = raw.strip()
  if cleaned.startswith("```json"):
    cleaned = cleaned[7:]
  if cleaned.startswith("```"):
    cleaned = cleaned[3:]
  if cleaned.endswith("```"):
    cleaned = cleaned[:-3]
  return cleaned.strip()

def agentic_processing(log:str,max_tries:int=3):
  curr = generate_extraction_prompt(log)

  attempt =1
  while attempt<=max_tries:
    print(f"[System] Attempt {attempt}/{max_tries}")

    # response = client.models.generate_content(model='gemini-1.5-flash',contents=curr)

    response = client.chat.completions.create(
      # model="llama3-8b-8192", decommissioned
      model = "openai/gpt-oss-20b",
      messages=[
          {"role":"system","content": "You are an automated backend infrastructure logs processing model."},
          {"role":"user","content": curr}
      ]
    )

    output = clean_json_string(response.choices[0].message.content)

    try:
      validated_data = SystemIncidentSchema.model_validate_json(output)
      print(f"Schema Validated on attempt: {attempt}")
      return validated_data, attempt
    except ValidationError as e:
      print(f"Formatting issue on attempt: {attempt}")

      error_trace = str(e)

      curr += f"""
      SYSTEM VALIDATION FAILED ON PREVIOUS ATTEMPT
      The JSON you just generated caused the following strict validation error:
      {error_trace}

      Read the error trace carefully. Fix the exact fields that failed validation to match the required schema constraints, and output the corrected JSON string.
      """

      attempt += 1

  print(f"FAILURE Agent exceeded number of tries")
  return None, attempt

In [204]:
print("Testing on log 1")
# test_incident_data, attempts = agentic_processing(unstructured_logs[0])
test_incident_data, attempts = agentic_processing(enterprise_logs_batch[0])

if test_incident_data:
  print("\nFinal Verified JSON Payload")
  print(test_incident_data.model_dump_json(indent=2))

Testing on log 1
[System] Attempt 1/3
Schema Validated on attempt: 1

Final Verified JSON Payload
{
  "incident_id": "INC-2026-00001",
  "severity_level": "MINOR",
  "main_reason": "JSON parsing error in dataset loading",
  "impacted_services": [],
  "est_downtime_min": 0
}


Batch Processing & Telemetry Dashboard  

In [205]:
import time

In [206]:
total_logs = len(enterprise_logs_batch)
success = 0
failure = 0
attempt_tracking = []
extracted_records = []

start_time = time.time()

for idx, log in enumerate(enterprise_logs_batch,1):
  print(f"\nProcessing log {idx}/{total_logs}")

  data,attempts = agentic_processing(log,max_tries=3)

  if data:
    success += 1
    extracted_records.append(data.model_dump())
    attempt_tracking.append(attempts)
  else:
    failure += 1
    attempt_tracking.append(-1)


end_time = time.time()

total_execution_time = round(end_time-start_time,2)


Processing log 1/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 2/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 3/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 4/30
[System] Attempt 1/3
Formatting issue on attempt: 1
[System] Attempt 2/3
Schema Validated on attempt: 2

Processing log 5/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 6/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 7/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 8/30
[System] Attempt 1/3
Formatting issue on attempt: 1
[System] Attempt 2/3
Schema Validated on attempt: 2

Processing log 9/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 10/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 11/30
[System] Attempt 1/3
Schema Validated on attempt: 1

Processing log 12/30
[System] Attempt 1/3
Formatting issue on attempt: 1
[System] Attempt 

In [207]:
print(f"Total logs processed: {total_logs}")
print(f"Successful Extractions: {success} ({success/total_logs*100:.1f}%)")
print(f"Failed Extractions: {failure}")
print(f"Total Pipeline execution time: {total_execution_time} seconds")


Total logs processed: 30
Successful Extractions: 30 (100.0%)
Failed Extractions: 0
Total Pipeline execution time: 171.44 seconds


In [208]:
for i,attempts in enumerate(attempt_tracking,1):
  if attempts != -1:
    print(f"Log - {i} Status: [SUCCESS] - Schema validated in {attempts} attempt(s).")
  else:
    print(f"Log - {i} Status: [FAILED] - Exceeded maximum tries.")

Log - 1 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 2 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 3 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 4 Status: [SUCCESS] - Schema validated in 2 attempt(s).
Log - 5 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 6 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 7 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 8 Status: [SUCCESS] - Schema validated in 2 attempt(s).
Log - 9 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 10 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 11 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 12 Status: [SUCCESS] - Schema validated in 2 attempt(s).
Log - 13 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 14 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 15 Status: [SUCCESS] - Schema validated in 1 attempt(s).
Log - 16 Status: [SUCCESS] - Schema validated in 1 attempt(s).
L

In [209]:
print("\n Sample of extracted JSON Array ready for Database Insertion:")
print(json.dumps(extracted_records[:6],indent=2))


 Sample of extracted JSON Array ready for Database Insertion:
[
  {
    "incident_id": "INC-2023-00001",
    "severity_level": "MINOR",
    "main_reason": "JSONDecodeError during dataset loading due to multiline JSON format",
    "impacted_services": [
      {
        "service_name": "DatasetLoader",
        "environment": "development"
      }
    ],
    "est_downtime_min": 0
  },
  {
    "incident_id": "INC-2026-00001",
    "severity_level": "MAJOR",
    "main_reason": "Dataset split sizes mismatch during loading causing NonMatchingSplitsSizesError.",
    "impacted_services": [
      {
        "service_name": "datasets",
        "environment": "development"
      }
    ],
    "est_downtime_min": 5
  },
  {
    "incident_id": "INC-2026-00001",
    "severity_level": "MINOR",
    "main_reason": "Dataset loader ignoring dummy folder causing data omission",
    "impacted_services": [
      {
        "service_name": "data_loader",
        "environment": "development"
      }
    ],
    "e